# BBC News Business Category Sub-classification
## NLP Pipeline: BGE-M3 + BERTopic (6 Modules)

**Architecture:**
- Layer 1: Data Ingestion — Raw .txt files → List[String]
- Layer 2: BGE-M3 Embeddings — 8192 token context, 1024 dims
- Layer 3: UMAP — Non-linear reduction 1024 → 5 dims
- Layer 4: HDBSCAN — Automatic density-based clustering
- Layer 5a: CountVectorizer — Removes noise words per cluster
- Layer 5b: c-TF-IDF — Distinctive keywords across clusters
- Layer 6: KeyBERTInspired — Semantic keyword refinement
- Layer 7: Output — DataFrame [filename, category, sub_category, preview]

**Final Output:** DataFrame mapping every article to its sub-category

## Environment Setup & Imports

In [ ]:
# Core libraries
import os
import re
import sys
import pandas as pd
import numpy as np

# Path setup — loader.py lives in parent directory
sys.path.append('..')
from loader import load_data

# Sentence Transformers — for BGE-M3
from sentence_transformers import SentenceTransformer

# BERTopic pipeline
from bertopic import BERTopic
from umap import UMAP
from hdbscan import HDBSCAN
from sklearn.feature_extraction.text import CountVectorizer
from bertopic.representation import KeyBERTInspired
from bertopic.vectorizers import ClassTfidfTransformer

# Visualization
import matplotlib.pyplot as plt

print("All imports successful")
print(f"Pandas version: {pd.__version__}")
print(f"NumPy version: {np.__version__}")

ModuleNotFoundError: No module named 'sentence_transformers'

## Data Ingestion Layer
### Loading raw business articles from local storage
### Source: BBC News dataset for business category
### No aggressive cleaning at this stage, BGE-M3 needs full sentence structure

In [ ]:
# Load all articles and labels
documents, labels = load_data('../data')

# Filter business articles only
business_docs = [documents[i] for i in range(len(documents)) 
                 if labels[i] == 'business']

# Store filenames for final output
business_files = [f"article_{i}.txt" for i in range(len(business_docs))]

print(f"Total articles in dataset: {len(documents)}")
print(f"Business articles: {len(business_docs)}")
print(f"\nSample article preview:")
print(f"{business_docs[0][:300]}")

Total articles in dataset: 2225
Business articles: 510

Sample article preview:
UK economy facing 'major risks'

The UK manufacturing sector will continue to face "serious challenges" over the next two years, the British Chamber of Commerce (BCC) has said.

The group's quarterly survey of companies found exports had picked up in the last three months of 2004 to their best level


## Light Cleaning (Data Normalization Layer)
### BGE-M3 understands full sentence structure, no aggressive cleaning needed
### We only remove noise that would confuse the transformer:
### - Encoding errors
### - Excessive whitespace
### Punctuation, stopwords, and proper nouns are intentionally preserved

In [ ]:
def light_clean(text):
    # Fix encoding errors
    text = text.encode('utf-8', 'ignore').decode('utf-8')
    
    # Remove excessive whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

# Apply light cleaning to all business articles
cleaned_business = [light_clean(doc) for doc in business_docs]

print(f"Cleaning complete — {len(cleaned_business)} articles processed")
print(f"\nBefore cleaning:")
print(f"{business_docs[0][:200]}")
print(f"\nAfter cleaning:")
print(f"{cleaned_business[0][:200]}")

Cleaning complete — 510 articles processed

Before cleaning:
UK economy facing 'major risks'

The UK manufacturing sector will continue to face "serious challenges" over the next two years, the British Chamber of Commerce (BCC) has said.

The group's quarterly 

After cleaning:
UK economy facing 'major risks' The UK manufacturing sector will continue to face "serious challenges" over the next two years, the British Chamber of Commerce (BCC) has said. The group's quarterly su


##  Deduplication
### Removing duplicate articles before embedding generation
### Identical documents produce identical vectors hurts clustering quality
### Preserving filename mapping for final DataFrame output

In [ ]:
# Remove duplicates while preserving filename mapping
seen = set()
unique_business = []
unique_files = []

for doc, filename in zip(cleaned_business, business_files):
    if doc not in seen:
        seen.add(doc)
        unique_business.append(doc)
        unique_files.append(filename)

print(f"Before deduplication: {len(cleaned_business)} articles")
print(f"After deduplication:  {len(unique_business)} articles")
print(f"Duplicates removed:   {len(cleaned_business) - len(unique_business)}")

Before deduplication: 510 articles
After deduplication:  503 articles
Duplicates removed:   7


## Embedding Layer (BGE-M3)
### Model: BAAI/bge-m3, 8192 token context window, 1024 dimensional output
### Mechanism: Full-sequence self-attention transformer
### Every article is read as one complete unit and mapped to a 1024-dim semantic vector
### Unlike bag-of-words approaches, BGE-M3 understands meaning, context, and intent

In [ ]:
# Load BGE-M3 model
print("Loading BGE-M3 model...")
print("This may take a few minutes on first load...")

model = SentenceTransformer('BAAI/bge-m3')

print("BGE-M3 model loaded successfully")
print(f"Max sequence length: {model.max_seq_length}")

Loading BGE-M3 model...
This may take a few minutes on first load...


pytorch_model.bin:   0%|          | 0.00/2.27G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.27G [00:00<?, ?B/s]

ValueError: Due to a serious vulnerability issue in `torch.load`, even with `weights_only=True`, we now require users to upgrade torch to at least v2.6 in order to use the function. This version restriction does not apply when loading files with safetensors.
See the vulnerability report here https://nvd.nist.gov/vuln/detail/CVE-2025-32434

##  Generate BGE-M3 Document Embeddings
### Each article is encoded as a single 1024-dimensional semantic vector
### BGE-M3 reads the entire document using full-sequence self-attention
### batch_size=32 processes 32 articles at a time to manage memory efficiently
### show_progress_bar=True lets us track encoding progress

In [ ]:
print("Generating BGE-M3 embeddings for 503 business articles...")
print("This will take a few minutes...")

embeddings = model.encode(
    unique_business,
    batch_size=32,
    show_progress_bar=True,
    normalize_embeddings=True
)

print(f"\nEmbeddings generated successfully")
print(f"Shape: {embeddings.shape}")
print(f"Each article → {embeddings.shape[1]} dimensional vector")

## BERTopic Pipeline (All 6 Modules)
### Module 3: UMAP — compresses 1024 dims to 5, random_state=42 for stability
### Module 4: HDBSCAN — finds natural sub-categories automatically
### Module 5a: CountVectorizer — removes noise words before keyword extraction
### Module 5b: c-TF-IDF — finds distinctive keywords per cluster
### Module 6: KeyBERTInspired — refines keywords using semantic similarity

In [ ]:
from bertopic.representation import KeyBERTInspired
from bertopic.vectorizers import ClassTfidfTransformer
from sklearn.feature_extraction.text import CountVectorizer
from umap import UMAP
from hdbscan import HDBSCAN

# Module 3 — UMAP
umap_model = UMAP(
    n_components=5,
    n_neighbors=15,
    min_dist=0.01,
    random_state=42
)

# Module 4 — HDBSCAN
hdbscan_model = HDBSCAN(
    min_cluster_size=10,  # Lowered to identify more, smaller clusters
    min_samples=3,        # Lowered to identify more, smaller clusters
    metric='euclidean',
    cluster_selection_method='eom'
)

# Module 5a — CountVectorizer
vectorizer_model = CountVectorizer(
    stop_words="english",
    max_df=0.8,
    ngram_range=(1, 2)
)

# Module 5b — c-TF-IDF
ctfidf_model = ClassTfidfTransformer(
    bm25_weighting=True
)

# Module 6 — KeyBERTInspired
representation_model = KeyBERTInspired()

# Assemble full BERTopic pipeline
topic_model = BERTopic(
    embedding_model=model,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    vectorizer_model=vectorizer_model,
    ctfidf_model=ctfidf_model,
    representation_model=representation_model,
    verbose=True
)

print("BERTopic pipeline assembled successfully")
print("All 6 modules configured")

## Fit BERTopic to BGE-M3 Embeddings
### Passing pre-computed BGE-M3 embeddings directly into BERTopic
### BERTopic runs all 6 modules sequentially:
### UMAP → HDBSCAN → CountVectorizer → c-TF-IDF → KeyBERTInspired
### topics_ contains sub-category assignment for every article
### probs_ contains confidence score for each assignment

In [ ]:
# Fit BERTopic using pre-computed BGE-M3 embeddings
print("Running BERTopic pipeline...")
print("UMAP → HDBSCAN → Vectorizer → c-TF-IDF → KeyBERT")
print("This may take a few minutes...")

topics, probs = topic_model.fit_transform(
    unique_business,
    embeddings=embeddings
)

print(f"\nBERTopic complete")
print(f"Total articles processed: {len(topics)}")
print(f"Sub-categories discovered: {len(set(topics)) - 1}")
print(f"Noise articles (topic -1): {topics.count(-1)}")